In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [2]:
class CausalSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, max_seq_len=128):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)

        # Create a static lower-triangular matrix (causal mask buffer)
        # tril creates: [[1, 0, 0], [1, 1, 0], [1, 1, 1]]
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len)).view(1, 1, max_seq_len, max_seq_len)
        self.register_buffer('causal_mask', mask)

    def forward(self, x):
        B, S, E = x.shape

        # 1. Project and split heads
        Q = x @ self.W_q.weight.T
        K = x @ self.W_k.weight.T
        V = x @ self.W_v.weight.T

        Q = Q.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.causal_mask[:, :, :S, :S] == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        attn_out = torch.matmul(weights, V)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, S, E)
        
        return self.out_proj(attn_out)

In [ ]:
# Quick Test
torch.manual_seed(42)
causal_layer = CausalSelfAttention(embed_dim=4, num_heads=1, max_seq_len=3)
test_input = torch.randn(1, 3, 4)
output = causal_layer(test_input)
print("Output shape:", output.shape)

# Exercise

In [9]:
import torch
import torch.nn.functional as F

SeqLen = 4

# Imagine these are raw attention scores computed for a 4-token sequence
# Shape: [SeqLen, SeqLen]
raw_scores = torch.tensor([[ 2.0,  4.0, -1.0,  0.5],
                           [ 1.5,  3.0,  5.2, -2.0],
                           [-0.5,  1.1,  2.2,  4.1],
                           [ 0.8, -1.5,  3.0,  1.2]])

# 1. Create a lower triangular mask of shape [4, 4] containing 1s and 0s
# Hint: torch.tril(torch.ones(SeqLen, SeqLen))
causal_mask = torch.tril(torch.ones(SeqLen, SeqLen))

# 2. Overwrite the raw scores where causal_mask == 0 with float('-inf')
masked_scores = raw_scores.masked_fill(
    causal_mask == 0,
    float('-inf')
    )

# 3. Apply softmax across the last dimension (dim=-1) to generate the weights
attention_weights = torch.softmax(masked_scores, dim=-1)

print("Causal Mask:\n", causal_mask)
print("\nMasked Scores:\n", masked_scores)
print("\nFinal Attention Weights:\n", torch.round(attention_weights, decimals=3))

Causal Mask:
 tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])

Masked Scores:
 tensor([[ 2.0000,    -inf,    -inf,    -inf],
        [ 1.5000,  3.0000,    -inf,    -inf],
        [-0.5000,  1.1000,  2.2000,    -inf],
        [ 0.8000, -1.5000,  3.0000,  1.2000]])

Final Attention Weights:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.1820, 0.8180, 0.0000, 0.0000],
        [0.0480, 0.2380, 0.7140, 0.0000],
        [0.0860, 0.0090, 0.7770, 0.1280]])
